# Indirect PIA base-paper reproduction — increment 1

This notebook only captures the Kaggle environment and validates the supplied datasets. It does **not** install dependencies or train a model. The official implementation files remain unchanged.

In [1]:
from pathlib import Path
import os
import subprocess

PROJECT_URL = "https://github.com/apusharma45/indirect-pia-detection.git"
PROJECT_DIR = Path("/kaggle/working/indirect-pia-detection")

if (PROJECT_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
elif PROJECT_DIR.exists() and any(PROJECT_DIR.iterdir()):
    raise RuntimeError(f"Refusing to clone into non-empty directory: {PROJECT_DIR}")
else:
    subprocess.run(["git", "clone", PROJECT_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
print("Project directory:", Path.cwd())
subprocess.run(["git", "rev-parse", "HEAD"], check=True)

Cloning into '/kaggle/working/indirect-pia-detection'...


Project directory: /kaggle/working/indirect-pia-detection
7f6a84545f09ed1f9c2b64bfa8a1315302cfc446


CompletedProcess(args=['git', 'rev-parse', 'HEAD'], returncode=0)

## Capture the untouched Kaggle environment

Run this before any package installation so we can decide whether the author's pinned environment is compatible with the assigned GPU image.

In [2]:
subprocess.run(
    ["python", "scripts/check_environment.py", "--output", "results/environment.txt"],
    check=True,
)

Indirect PIA reproduction environment
generated_at_utc: 2026-10-07T16:06:03.460142+00:00
git_commit: 7f6a84545f09ed1f9c2b64bfa8a1315302cfc446
python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
python_executable: /usr/bin/python3
platform: Linux-6.18.48+-x86_64-with-glibc2.39
kaggle_kernel_run_type: Interactive
cuda_visible_devices: not set

Package versions
----------------
torch: 2.11.0+cu128
transformers: 5.16.1
deepspeed: not installed
accelerate: 1.14.0
datasets: 4.8.5
tokenizers: 0.23.1

PyTorch / CUDA
--------------
import: ok
version: 2.11.0+cu128
cuda_available: True
compiled_cuda_version: 12.8
cudnn_version: 91900
device_count: 2
bf16_supported: True
devices: [{'index': 0, 'name': 'Tesla T4', 'total_memory_bytes': 15636037632, 'total_memory_gib': 14.56, 'compute_capability': '7.5'}, {'index': 1, 'name': 'Tesla T4', 'total_memory_bytes': 15636037632, 'total_memory_gib': 14.56, 'compute_capability': '7.5'}]

nvidia-smi
----------
0, Tesla T4, 15360 MiB, 580.178.04, 7.5
1

CompletedProcess(args=['python', 'scripts/check_environment.py', '--output', 'results/environment.txt'], returncode=0)

## Validate the supplied data

This reads and hashes all JSON datasets and regenerates the two summaries. It does not alter any source data.

In [3]:
subprocess.run(
    ["python", "scripts/inspect_data.py", "--data-dir", "data", "--output-dir", "results"],
    check=True,
)
print((PROJECT_DIR / "results/data_summary.md").read_text(encoding="utf-8"))

Wrote /kaggle/working/indirect-pia-detection/results/data_summary.json
Wrote /kaggle/working/indirect-pia-detection/results/data_summary.md
Status: ok; files: 5; examples: 58848
# Dataset inspection summary

- Status: `ok`
- Data directory: `data`
- Git revision: `7f6a84545f09ed1f9c2b64bfa8a1315302cfc446`
- JSON files: 5
- Total examples: 58848

The script only read and hashed the source datasets; it did not modify them.

## `crafted_instruction_data_alpaca.json`

- Size: 11,460,379 bytes
- SHA-256: `f72822723a36faa3fccc22ff474c61f149533d5899519de153abd4307f986ae3`
- Top level: array
- Examples: 19,157
- Keys: `input`, `instruction`, `output`
- Malformed/non-object records: 0
- Records missing modal fields: 0
- Label fields: `{}`
- Position fields: `{}`
- Attack/category fields: `{}`

Representative records (text is truncated):

```json
[
  {
    "index": 0,
    "record": {
      "input": "4/16",
      "instruction": "Explain why the following fraction is equivalent to 1/4",
      "out

## Return the evidence through GitHub

Review the reports first. Set `PUSH_RESULTS = True` only after Git authentication is configured in Kaggle. No token is stored or printed by this notebook.

In [4]:
RETURN_FILES = [
    "results/environment.txt",
    "results/data_summary.json",
    "results/data_summary.md",
]
subprocess.run(["git", "status", "--short", "--", *RETURN_FILES], check=True)

PUSH_RESULTS = False
if PUSH_RESULTS:
    subprocess.run(["git", "add", "--", *RETURN_FILES], check=True)
    staged = subprocess.run(["git", "diff", "--cached", "--quiet"])
    if staged.returncode == 1:
        subprocess.run(["git", "commit", "-m", "chore: record kaggle environment and data audit"], check=True)
        subprocess.run(["git", "push", "origin", "main"], check=True)
    elif staged.returncode == 0:
        print("No report changes to commit.")
    else:
        raise RuntimeError(f"git diff failed with exit code {staged.returncode}")
else:
    print("Reports are ready. Review them, configure Git authentication, then set PUSH_RESULTS=True and rerun this cell.")

 M results/data_summary.json
 M results/data_summary.md
?? results/environment.txt
Reports are ready. Review them, configure Git authentication, then set PUSH_RESULTS=True and rerun this cell.


## Increment 2 — DeBERTa smoke test

This is a bounded engineering test through the authors' unmodified `train_head.py`. It uses 16 SQuAD contexts and is not a scientific result.

In [ ]:
# Refresh the cloned repository, then run the smoke test on GPU 0.
subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
os.chdir(PROJECT_DIR)
smoke = subprocess.run(["bash", "scripts/run_deberta_smoke.sh", "--install-deps"])
print("Smoke runner return code:", smoke.returncode)


In [ ]:
# Print the evidence needed for the next diagnosis.
RESULT_DIR = PROJECT_DIR / "results/deberta_smoke"
for name in ["metrics.json", "config.json", "error.txt", "environment.txt", "command.txt"]:
    result_path = RESULT_DIR / name
    if result_path.exists():
        print(f"\n===== {name} =====")
        print(result_path.read_text(encoding="utf-8", errors="replace"))

for name in ["dependency_install_log.txt", "training_log.txt"]:
    log_path = RESULT_DIR / name
    if log_path.exists():
        print(f"\n===== final 120 lines: {name} =====")
        lines = log_path.read_text(encoding="utf-8", errors="replace").splitlines(True)
        print("".join(lines[-120:]))


## Return this smoke result

After the two cells above finish, use Kaggle's **Save Version** and export/update this same notebook on GitHub. A `GITHUB_TOKEN` is not required for this smoke increment because the structured result, error, environment, and log tails are printed into the notebook output. After pulling the updated notebook locally, Codex will recover and inspect that evidence before preparing any full training run.
